# 🌐 Прогноз RPS по endpoint-ам — Ridge + Walk-forward (Схема 2)

Методика — `cpu_walkforward.ipynb`. CatBoost заменён на **Ridge** с теми же признаками. Для каждого handler-а обучается **отдельный** Ridge + StandardScaler (масштабы RPS у разных эндпоинтов сильно различаются), но используется единый набор фич — это полностью совместимо с `RidgeFeatureBuilder` в `app/feature_builder.py`.

Итоговый артефакт — единый `joblib`-bundle с per-handler моделями и таблицей `handler_mapping`.

## 1. Настройка окружения

In [ ]:
import warnings; warnings.filterwarnings('ignore')

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from statsmodels.tsa.holtwinters import ExponentialSmoothing

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

# ── Единый тёмный стиль ───────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 110, 'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22', 'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9', 'axes.titlecolor': '#e6edf3',
    'axes.titlesize': 12, 'axes.labelsize': 10,
    'axes.grid': True, 'grid.color': '#21262d', 'grid.linewidth': 0.7,
    'xtick.color': '#8b949e', 'ytick.color': '#8b949e',
    'legend.facecolor': '#161b22', 'legend.edgecolor': '#30363d',
    'legend.labelcolor': '#c9d1d9', 'text.color': '#c9d1d9',
    'lines.linewidth': 1.5, 'font.family': 'DejaVu Sans',
})

C = dict(actual='#58a6ff', ridge='#f78166', naive='#7ee787',
         hw='#d2a8ff', lasso='#ffa657', train='#388bfd22',
         test='#f7816622', ci='#f0883e')

FREQ_SEC     = 15
HORIZONS_MIN = [1, 5, 10, 15, 30, 45, 60]
print("✅ Окружение готово")

## 2. Загрузка RPS по всем endpoint-ам

In [ ]:
import os

RPS_FILES = [
    '../data/users__rps.csv',
    '../data/accounts__rps.csv',
    '../data/transfers__rps.csv',
]
dfs = []
for f in RPS_FILES:
    if os.path.exists(f):
        d = pd.read_csv(f)
        dfs.append(d)
        print(f"Loaded {f}: {d.shape}")

raw = pd.concat(dfs, ignore_index=True)
raw['datetime'] = pd.to_datetime(raw['timestamp'], unit='s')
raw = raw.sort_values(['handler', 'datetime']).reset_index(drop=True)

HANDLERS = sorted(raw['handler'].unique().tolist())
print(f"\nHandlers: {HANDLERS}")

# Каждый handler — собственный ряд с шагом 15 с
series_by_handler: dict[str, pd.Series] = {}
for h in HANDLERS:
    sub = raw[raw['handler'] == h].drop_duplicates('timestamp')
    sub = sub.set_index('datetime')['value']
    full_idx = pd.date_range(sub.index.min(), sub.index.max(), freq='15s')
    sub = sub.reindex(full_idx).interpolate('time').fillna(0.0).clip(lower=0.0)
    sub.index.name = 'datetime'
    series_by_handler[h] = sub
    print(f"  {h:32s}  rows={len(sub):>7}  "
          f"mean={sub.mean():.2f}  max={sub.max():.2f}")

## 3. Признаки для Ridge (общие на все handler-ы)

In [ ]:
def make_features(series, extra_cols: dict | None = None):
    """
    Признаки для Ridge — копия `make_features` из cpu_walkforward.ipynb.
    extra_cols (dict[str, pd.Series]) — дополнительные колонки (например, one-hot
    handler-а для RPS); индексируются тем же индексом, что и series.
    """
    f = pd.DataFrame(index=series.index)
    for lag in [1, 2, 4, 8, 16, 32, 60, 120, 240]:
        f[f'lag_{lag}'] = series.shift(lag)
    for w in [4, 20, 60, 120, 240]:
        f[f'roll_mean_{w}'] = series.shift(1).rolling(w).mean()
        f[f'roll_std_{w}']  = series.shift(1).rolling(w).std()
    for span in [4, 20, 60]:
        f[f'ewm_{span}'] = series.shift(1).ewm(span=span).mean()
    f['hour_sin']   = np.sin(2*np.pi*series.index.hour/24)
    f['hour_cos']   = np.cos(2*np.pi*series.index.hour/24)
    f['minute_sin'] = np.sin(2*np.pi*series.index.minute/60)
    f['minute_cos'] = np.cos(2*np.pi*series.index.minute/60)
    f['diff_1']  = series.diff(1).shift(1)
    f['diff_20'] = series.diff(20).shift(1)
    if extra_cols:
        for k, v in extra_cols.items():
            f[k] = v.reindex(f.index).values
    return f

BASE_FEATURE_COLS = (
    [f'lag_{l}'       for l in [1, 2, 4, 8, 16, 32, 60, 120, 240]]
    + [f'roll_mean_{w}' for w in [4, 20, 60, 120, 240]]
    + [f'roll_std_{w}'  for w in [4, 20, 60, 120, 240]]
    + [f'ewm_{s}'       for s in [4, 20, 60]]
    + ['hour_sin', 'hour_cos', 'minute_sin', 'minute_cos', 'diff_1', 'diff_20']
)
LAGS         = [1, 2, 4, 8, 16, 32, 60, 120, 240]
ROLL_WINDOWS = [4, 20, 60, 120, 240]
EWM_SPANS    = [4, 20, 60]
MIN_HISTORY_POINTS = max(max(LAGS), max(ROLL_WINDOWS)) + 1   # 241
ALPHA_RIDGE  = 10.0

print(f"Признаков без extra: {len(BASE_FEATURE_COLS)}")

## 4. Walk-forward (Схема 2) — по каждому endpoint-у

In [ ]:
def walk_forward_eval(series, horizon_min, n_passes=5,
                      train_frac=0.7, alpha_ridge=10.0,
                      extra_cols: dict | None = None,
                      feature_cols: list | None = None,
                      do_hw: bool = True, hw_period: int = 240):
    """
    Схема 2 (walk-forward, скользящее окно фиксированного размера).
    Возвращает усреднённые метрики Ridge / Naive / Holt-Winters.
    """
    h = int(horizon_min * 60 / FREQ_SEC)
    N = len(series)
    train_size = int(N * train_frac)

    delta = h
    max_passes = (N - train_size) // delta
    K = min(n_passes, max_passes)
    if K == 0:
        return None

    fcols = feature_cols or BASE_FEATURE_COLS
    X_all = make_features(series, extra_cols=extra_cols)[fcols]
    y_all_shift = series.shift(-h).rename('target')
    data = pd.concat([X_all, y_all_shift], axis=1).dropna()

    ridge_metrics, naive_metrics, hw_metrics = [], [], []
    all_preds = []
    y_arr = series.values

    for k in range(K):
        train_start = k * delta
        train_end   = train_start + train_size
        test_end    = train_end + delta
        if test_end > N:
            break

        tr_data = data.iloc[train_start:train_end].dropna()
        te_data = data.iloc[train_end:test_end].dropna()
        if len(tr_data) < 50 or len(te_data) == 0:
            continue

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(tr_data.drop(columns='target'))
        Xte = scaler.transform(te_data.drop(columns='target'))

        ridge = Ridge(alpha=alpha_ridge)
        ridge.fit(Xtr, tr_data['target'])
        y_pred_r = ridge.predict(Xte)
        y_true   = te_data['target'].values

        # Naive baseline — predict-last-known-window-shift
        naive_te = y_arr[train_end - h:test_end - h][:len(y_true)]

        # Holt-Winters
        if do_hw:
            try:
                hw_train = series.iloc[train_start:train_end]
                hw_m = ExponentialSmoothing(
                    hw_train, seasonal='add', seasonal_periods=hw_period, trend='add'
                ).fit(optimized=True)
                hw_fc = np.asarray(hw_m.forecast(delta))[:len(y_true)]
            except Exception:
                hw_fc = naive_te
        else:
            hw_fc = naive_te

        def mets(yt, yp):
            yt, yp = np.asarray(yt), np.asarray(yp)
            mask = np.isfinite(yt) & np.isfinite(yp)
            yt, yp = yt[mask], yp[mask]
            if len(yt) == 0:
                return dict(MAE=np.nan, RMSE=np.nan, R2=np.nan)
            return dict(
                MAE=mean_absolute_error(yt, yp),
                RMSE=np.sqrt(mean_squared_error(yt, yp)),
                R2=r2_score(yt, yp),
            )

        ridge_metrics.append(mets(y_true, y_pred_r))
        naive_metrics.append(mets(y_true, naive_te))
        hw_metrics.append(mets(y_true, hw_fc))

        te_times = series.index[train_end:test_end]
        all_preds.append(dict(
            k=k+1, times=te_times[:len(y_true)],
            y_true=y_true, ridge=y_pred_r, naive=naive_te, hw=np.asarray(hw_fc),
        ))

    def avg(mlist):
        df = pd.DataFrame(mlist)
        return df[['MAE','RMSE','R2']].mean()

    return dict(
        ridge=avg(ridge_metrics), naive=avg(naive_metrics), hw=avg(hw_metrics),
        passes=all_preds, K=len(ridge_metrics),
    )

print("Функция walk_forward_eval готова ✅  (метрика: rps)")

In [ ]:
print("Запуск walk-forward оценки RPS по каждому endpoint-у ...")

# Для каждого handler-а — отдельная walk-forward таблица
all_results: dict[str, dict] = {}

for h_name, series in series_by_handler.items():
    print(f"\n── handler: {h_name} ──")
    wf_results = {}
    for h_min in HORIZONS_MIN:
        # HW отключаем для коротких RPS-рядов / экономим время на длинных
        res = walk_forward_eval(series, h_min, n_passes=5, train_frac=0.70,
                                alpha_ridge=ALPHA_RIDGE, do_hw=True)
        wf_results[h_min] = res
        if res:
            print(f"  {h_min:>3} мин  K={res['K']}  "
                  f"Ridge MAE={res['ridge']['MAE']:.3f}  "
                  f"Naive MAE={res['naive']['MAE']:.3f}  "
                  f"HW MAE={res['hw']['MAE']:.3f}")
    all_results[h_name] = wf_results

print("\n✅ Walk-forward завершён по всем endpoint-ам")

## 5. Сравнение моделей по horizon × handler

In [ ]:
rows = []
for h_name, wf_results in all_results.items():
    for h_min in HORIZONS_MIN:
        res = wf_results.get(h_min)
        if res is None:
            continue
        rows.append({
            'handler':       h_name,
            'Горизонт, мин': h_min,
            'Ridge MAE':     res['ridge']['MAE'],
            'Naive MAE':     res['naive']['MAE'],
            'HW MAE':        res['hw']['MAE'],
            'Ridge R²':      res['ridge']['R2'],
        })

cmp = pd.DataFrame(rows)
print("=== Сравнительная таблица по handler×горизонт ===")
display(cmp.round(3))

# График Ridge MAE по handler-ам и горизонтам
fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('#0d1117'); ax.set_facecolor('#161b22')
hlist = [h for h in HORIZONS_MIN if any(all_results[hn].get(h) for hn in HANDLERS)]
x = np.arange(len(hlist))
width = 0.8 / max(1, len(HANDLERS))
palette = [C['ridge'], C['hw'], C['lasso'], C['naive'], C['actual']]
for i, h_name in enumerate(HANDLERS):
    vals = [all_results[h_name][h]['ridge']['MAE'] if all_results[h_name].get(h) else np.nan
            for h in hlist]
    ax.bar(x + i*width, vals, width, color=palette[i % len(palette)],
           alpha=0.85, label=h_name)
ax.set_xticks(x + width*(len(HANDLERS)-1)/2)
ax.set_xticklabels([f'{h}м' for h in hlist])
ax.set_xlabel('Горизонт'); ax.set_ylabel('Ridge MAE')
ax.set_title('Ridge MAE по handler-ам и горизонтам')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## 6. Финальное обучение Ridge per-handler и сохранение
Фиксируем горизонт 5 мин. Для каждого handler-а — собственная пара (Ridge, StandardScaler), всё кладётся в один `joblib`-bundle.

In [ ]:
FINAL_HORIZON_MIN   = 5
FINAL_HORIZON_STEPS = int(FINAL_HORIZON_MIN * 60 / FREQ_SEC)
FEATURE_COLS = list(BASE_FEATURE_COLS)

# Для каждого handler-а — отдельный Ridge + Scaler.
# RPS-ряды очень разные по масштабу (см. сводку выше), поэтому общий scaler
# был бы плохой идеей. Отдельные модели:
#   * сохраняют единый набор фич (совместимость с RidgeFeatureBuilder),
#   * корректно учитывают per-endpoint масштаб через свой StandardScaler.

handler_models: dict[str, dict] = {}
per_handler_metrics: dict[str, dict] = {}

for h_name, series in series_by_handler.items():
    X_full = make_features(series)[FEATURE_COLS]
    y_full = series.shift(-FINAL_HORIZON_STEPS).rename('target')
    data_full = pd.concat([X_full, y_full], axis=1).dropna()
    if len(data_full) < 200:
        print(f"⚠️  {h_name}: слишком мало точек ({len(data_full)}), пропускаем")
        continue

    X_train = data_full[FEATURE_COLS].values
    y_train = data_full['target'].values

    scaler = StandardScaler()
    Xs = scaler.fit_transform(X_train)
    ridge = Ridge(alpha=ALPHA_RIDGE).fit(Xs, y_train)
    y_pred_in = ridge.predict(Xs)

    in_sample = dict(
        MAE=float(mean_absolute_error(y_train, y_pred_in)),
        RMSE=float(np.sqrt(mean_squared_error(y_train, y_pred_in))),
        R2=float(r2_score(y_train, y_pred_in)),
    )
    per_handler_metrics[h_name] = in_sample
    print(f"{h_name:32s}  in-sample MAE={in_sample['MAE']:.3f}  R²={in_sample['R2']:.4f}")

    handler_models[h_name] = dict(model=ridge, scaler=scaler)

# ── Сохранение ─────────────────────────────────────────────────────────
MODELS_DIR  = Path('../models'); MODELS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH  = MODELS_DIR / 'rps_forecast_model.joblib'
CONFIG_PATH = MODELS_DIR / 'rps_model_config.json'

handler_mapping = {h: i for i, h in enumerate(HANDLERS)}

bundle = {
    # Совместимость с RidgePredictor: общий model/scaler = модель первого handler-а
    # как дефолтная (если handler неизвестен инференсу). Основное хранилище —
    # 'handlers' с per-endpoint моделями.
    'model':  next(iter(handler_models.values()))['model']  if handler_models else None,
    'scaler': next(iter(handler_models.values()))['scaler'] if handler_models else None,
    'feature_cols': FEATURE_COLS,
    'handlers': handler_models,
    'handler_mapping': handler_mapping,
    'meta': {
        'model_type': 'ridge',
        'metric_type': 'rps',
        'per_handler': True,
        'forecast_horizon_steps': FINAL_HORIZON_STEPS,
        'forecast_horizon_seconds': FINAL_HORIZON_STEPS * FREQ_SEC,
        'scrape_interval_sec': FREQ_SEC,
        'lags': LAGS,
        'rolling_windows': ROLL_WINDOWS,
        'ewm_spans': EWM_SPANS,
        'min_history_points': MIN_HISTORY_POINTS,
        'alpha': ALPHA_RIDGE,
    },
}
joblib.dump(bundle, MODEL_PATH)
print(f"\n✅ Модель сохранена: {MODEL_PATH.resolve()}")

config_json = {
    'model_type': 'ridge',
    'metric_type': 'rps',
    'per_handler': True,
    'feature_cols': FEATURE_COLS,
    'categorical_features': [],
    'handler_mapping': handler_mapping,
    'forecast_horizon_steps': FINAL_HORIZON_STEPS,
    'forecast_horizon_seconds': FINAL_HORIZON_STEPS * FREQ_SEC,
    'scrape_interval_sec': FREQ_SEC,
    'lags': LAGS,
    'rolling_windows': ROLL_WINDOWS,
    'ewm_spans': EWM_SPANS,
    'min_history_points': MIN_HISTORY_POINTS,
    'alpha': ALPHA_RIDGE,
    'per_handler_in_sample': per_handler_metrics,
    'walk_forward_5min': {
        h_name: (
            {k: float(v) for k, v in all_results[h_name][5]['ridge'].items()}
            if all_results.get(h_name, {}).get(5) else None
        )
        for h_name in HANDLERS
    },
}
with open(CONFIG_PATH, 'w') as f:
    json.dump(config_json, f, indent=2, ensure_ascii=False)
print(f"✅ Конфиг сохранён:  {CONFIG_PATH.resolve()}")